# 一个 body 就够：拆掉 `_trimmed_*.urdf`

<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/notebooks/pybullet_egl_mask_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

`PyBulletRenderer` 以前同时加载**两台机器人**：pybullet_data 的通用 Franka（藏掉 hand/finger）
和仓库自己的 Robotiq URDF（藏掉整条 `panda_link*` 手臂），两边各藏一半拼成一台。
"藏"用的是 `changeVisualShape(rgbaColor=[0,0,0,0])` —— 一个只有 CPU 光栅化器认的约定。
换到 EGL 之后 alpha 被无视，那条本该隐形的手臂挡在真手臂前面，mask 塌掉 12 倍。
当时的修法是复制一份 URDF、删掉几何、写成 `_trimmed_<hash>.urdf` 再加载。

这个 notebook 说明那份复制根本不必要：**`franka_panda_robotiq_2f85_og.urdf` 本身就是整台机器人**，
第二台 body 是多余的。去掉它，隐藏机制、trim、临时文件一起消失。

| | mask IoU（对 CPU 基线） | 对实测深度的残差中位数 | p90 |
|---|---|---|---|
| 双 body + `_trimmed_*.urdf`（旧） | 0.9938 | 3.84 mm | 18.39 mm |
| 单 body + `URDF_IGNORE_COLLISION_SHAPES`（新） | 0.9811 | **3.16 mm** | **14.77 mm** |

IoU 掉一点不是变差 —— 是手臂网格换了来源：从 pybullet_data 的通用模型，换成仓库自己策管的
DROID 网格。**用实测深度当裁判，新版更准**，而这还是在偏向旧版的条件下测的（现有外参就是用旧模型优化的）。

---

**需要 GPU**（EGL 加载不上就没有对照），以及一个真实 episode 的 Stage 1/2 输出。

## 0. 环境

In [ ]:
import importlib.util
import os
import subprocess
import sys

IN_COLAB = (
  importlib.util.find_spec("google") is not None
  and importlib.util.find_spec("google.colab") is not None
)

if IN_COLAB:
  REPO_DIR = "/content/droid"
  if not os.path.exists(REPO_DIR):
    subprocess.run(
      ["git", "clone", "--recursive", "https://github.com/yangyi02/droid.git", REPO_DIR], check=True
    )
  os.chdir(REPO_DIR)
else:
  REPO_DIR = os.getcwd()
  while not os.path.exists(os.path.join(REPO_DIR, "core", "physics.py")):
    parent = os.path.dirname(REPO_DIR)
    if parent == REPO_DIR:
      raise SystemExit("open this notebook from the droid checkout")
    REPO_DIR = parent
if REPO_DIR not in sys.path:
  sys.path.insert(0, REPO_DIR)

import pybullet

import core.physics
from config import get_config

config = get_config()
URDF = config.paths.urdf

pybullet.connect(pybullet.DIRECT)
EGL_OK = core.physics._load_egl()
pybullet.disconnect()

print("numpy support :", bool(pybullet.isNumpyEnabled()))
print("EGL plugin    :", EGL_OK)
if not EGL_OK:
  print("\n没有 GPU 光栅化器，下面 EGL 的那两栏会退回 CPU，对照不成立。")

## 1. 数据：场景相机，不是腕部相机

这一点值得单独说：这个 episode 里 `sorted(cameras)[0]` 正好是**腕部相机**，
而腕部视角里手臂基本不在画面内，mask 几乎只剩夹爪 —— 在那台相机上比 mask 是量不出手臂差异的。
下面显式取一台非腕部相机。

In [ ]:
import glob

import numpy as np

import core.io

DEPTH_ROOT = os.path.join(REPO_DIR, "data", "cache", "depth")
EXT_ROOT = os.path.join(REPO_DIR, "data", "cache", "extrinsics")
N_FRAMES = 16

ready = sorted(
  os.path.basename(d)
  for d in glob.glob(os.path.join(DEPTH_ROOT, "*"))
  if os.path.exists(os.path.join(d, "robot.npz"))
  and glob.glob(os.path.join(EXT_ROOT, os.path.basename(d), "*", "extrinsics.json"))
)
if not ready:
  raise SystemExit(f"no episode with both depth and extrinsics under {DEPTH_ROOT}")
EPISODE_ID = ready[0]

scene = core.io.load_depth_data(EPISODE_ID, DEPTH_ROOT, load_video=None)
state = core.io.load_extrinsics(scene, EXT_ROOT)

wrist = scene["meta"]["wrist_serial"]
CAM = sorted(c for c in scene["camera"] if c != wrist)[0]

obs_all = scene["camera"][CAM]["raw_depth"]
take = np.linspace(0, len(obs_all) - 1, N_FRAMES).astype(int)

K = scene["camera"][CAM]["K_mat"]
EXT = state[CAM]["extrinsics"][take]
JOINTS = scene["robot"]["joint_positions"][take]
GRIP = scene["robot"]["gripper_positions"][take]
OBS = obs_all[take].astype(np.float32)
H, W = obs_all.shape[1], obs_all.shape[2]

print(f"episode : {EPISODE_ID}")
print(f"camera  : {CAM}   (wrist is {wrist}, deliberately not used)")
print(f"frames  : {N_FRAMES} of {len(obs_all)}, spread over the episode   |   {W}x{H}")

## 2. 旧设计长什么样

不在这里重抄一遍 —— 直接从它还活着的那个 commit 把 `core/physics.py` 读出来跑。

In [ ]:
import types

OLD_REV = "acb54b6"  # the last commit with the two-body renderer

src = subprocess.run(
  ["git", "show", f"{OLD_REV}:core/physics.py"], capture_output=True, text=True, check=True
).stdout
old_physics = types.ModuleType("old_physics")
exec(compile(src, f"core/physics.py@{OLD_REV}", "exec"), old_physics.__dict__)

for f in glob.glob(os.path.join(REPO_DIR, "assets", "**", "_trimmed_*.urdf"), recursive=True):
  os.remove(f)

old = old_physics.PyBulletRenderer(URDF, gpu=True)  # gpu=True is what writes the trimmed copies
print(f"bodies loaded      : {pybullet.getNumBodies()}")
print(f"  robot id={old.robot_id}  hidden links {old.hidden_robot_links}")
print(f"  ghost id={old.ghost_id}  hidden links {old.hidden_ghost_links}")
trimmed = glob.glob(os.path.join(REPO_DIR, "assets", "**", "_trimmed_*.urdf"), recursive=True)
print("\ntrimmed copies written next to the source URDFs:")
for f in sorted(trimmed):
  print("  ", os.path.relpath(f, REPO_DIR))

两台机器人，两份隐藏名单。`gpu=True` 时这两份名单还要各自变成一个 `_trimmed_<hash>.urdf`
写到磁盘上 —— 因为 EGL 不认 `alpha=0`，只能让那些 link 真的没有几何可画。

## 3. 但那个 URDF 本来就是整台机器人

被当成"ghost"的那个文件里有什么：

In [ ]:
import xml.etree.ElementTree as ET

links = ET.parse(URDF).getroot().findall("link")
sc = [l for l in links if l.get("name").endswith("_sc")]
arm = [l for l in links if l.get("name").startswith("panda_link") and l not in sc]
rest = [l for l in links if l not in arm and l not in sc]

def count(group, tag):
  return sum(len(l.findall(tag)) for l in group)


print(f"{len(links)} links:")
print(f"  {len(arm):2d} panda_link*      visual {count(arm, 'visual'):2d}")
print(f"  {len(sc):2d} panda_link*_sc   visual {count(sc, 'visual'):2d}"
      f"   collision {count(sc, 'collision')}")
print(f"  {len(rest):2d} gripper + mount  visual {count(rest, 'visual'):2d}")
print("\n它自己就带整条手臂，第二台 body 是多余的。")

## 4. 唯一的坑：`_sc` 自碰撞链路

上面那 16 条 `panda_link*_sc` **有 collision、没有 visual**。
pybullet 在一个 link 没有 visual 时会拿它的 collision 几何来画，所以直接单 body 加载，
手臂上会套一圈碰撞胶囊。`URDF_IGNORE_COLLISION_SHAPES` 一个 flag 就够：

In [ ]:
def drawn_links(flags):
  pybullet.resetSimulation()
  body = pybullet.loadURDF(URDF, useFixedBase=True, flags=flags)
  info = [pybullet.getJointInfo(body, i) for i in range(pybullet.getNumJoints(body))]
  arm = [j[0] for j in info if "panda_joint" in j[1].decode() and j[2] != pybullet.JOINT_FIXED]
  for i, angle in zip(arm, JOINTS[0]):
    pybullet.resetJointState(body, i, angle)

  cam = EXT[0][:3, 3]
  view = pybullet.computeViewMatrix(
    cam.tolist(), (cam + EXT[0][:3, 2]).tolist(), (-EXT[0][:3, 1]).tolist()
  )
  proj = core.physics.PyBulletRenderer._get_projection_matrix(None, K, W, H)
  _, _, _, _, seg = pybullet.getCameraImage(
    W, H, viewMatrix=view, projectionMatrix=proj,
    renderer=pybullet.ER_BULLET_HARDWARE_OPENGL,
    flags=pybullet.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX,
  )
  seg = np.reshape(seg, (H, W)).astype(np.int32)
  names = {i: info[i][12].decode() for i in range(len(info))}
  names[-1] = pybullet.getBodyInfo(body)[0].decode()
  on_screen = (seg & 0xFFFFFF) == body
  return sorted({names[i] for i in np.unique(((seg >> 24) - 1)[on_screen]).tolist()}), on_screen


pybullet.disconnect()
pybullet.connect(pybullet.DIRECT)
core.physics._load_egl()

for label, flags in [
  ("default", 0),
  ("URDF_IGNORE_COLLISION_SHAPES", pybullet.URDF_IGNORE_COLLISION_SHAPES),
]:
  names, on_screen = drawn_links(flags)
  capsules = [n for n in names if n.endswith("_sc")]
  print(f"{label}")
  print(f"  {100 * on_screen.mean():5.2f}% of frame, {len(names)} links,"
        f" {len(capsules)} of them _sc")
  print(f"  {names}\n")
pybullet.disconnect()

## 5. 三种配置，同一台相机同一批帧

In [ ]:
def run(build):
  r = build()
  masks, depths = [], []
  for t in range(N_FRAMES):
    r.update_robot_pose(JOINTS[t], GRIP[t])
    masks.append(r.render_mask(EXT[t], K, W, H))
    depths.append(r.render_depth(EXT[t], K, W, H))
  return np.array(masks), np.array(depths, dtype=np.float32)


RES = {
  "old, CPU (alpha=0)": run(lambda: old_physics.PyBulletRenderer(URDF, gpu=False)),
  "old, EGL (trimmed urdf)": run(lambda: old_physics.PyBulletRenderer(URDF, gpu=True)),
  "new, EGL (one body)": run(lambda: core.physics.PyBulletRenderer(URDF, gpu=True)),
}

base_m, base_d = RES["old, CPU (alpha=0)"]
print(f"{'':28s}{'mask px':>10s}{'IoU vs CPU':>13s}{'depth med':>13s}{'>1.5cm':>9s}")
for name, (m, d) in RES.items():
  iou = (base_m & m).sum() / (base_m | m).sum()
  both = (base_d > 0) & (d > 0)
  diff = np.abs(base_d - d)[both]
  print(f"{name:28s}{m.sum(axis=(1, 2)).mean():>10.0f}{iou:>13.4f}"
        f"{np.median(diff):>11.1e} m{100 * (diff > 1.5e-2).mean():>8.2f}%")

新版和旧版差在哪：手臂网格的来源。旧版用 pybullet_data 那台通用 Franka 的手臂，
新版用仓库 `assets/franka_description/meshes/visual/link*.dae`。
所以 IoU 不是 1.0 —— 问题是哪个更像真机器人。

## 6. 让实测深度当裁判

渲染出来的机器人表面，和这台相机自己测到的深度差多少。越小越接近真机。

In [ ]:
print("robot-surface depth residual against the camera's own measurement")
print("(the extrinsics were optimised with the old model, so this favours it)\n")
for name in ("old, EGL (trimmed urdf)", "new, EGL (one body)"):
  m, d = RES[name]
  sel = m & (d > 0) & (OBS > 0)
  resid = np.abs(d - OBS)[sel]
  print(f"  {name:26s} median {1000 * np.median(resid):5.2f} mm"
        f"   p75 {1000 * np.percentile(resid, 75):5.2f}"
        f"   p90 {1000 * np.percentile(resid, 90):6.2f}")

## 7. 结论

单 body 更简单，而且更准。`core/physics.py` 里跟着消失的东西：

| 删掉的 | 为什么不再需要 |
|---|---|
| `_trimmed_urdf()` + `hashlib` / `ElementTree` | 没有要删几何的 link 了 |
| `_hidden_on_arm` / `_hidden_on_ghost` | 没有要藏的 link 了 |
| `hidden_robot_links` / `hidden_ghost_links` 及其构造循环 | 同上 |
| `changeVisualShape(rgbaColor=[0,0,0,0])` | 这就是 EGL 不认的那个约定 |
| `ghost_id` / `ghost_arm_joints` / 第二次 `loadURDF` | 只剩一台机器人 |
| `import pybullet_data` | URDF 在 `assets/` 里，网格按相对路径解析 |

`render_mask` 从"两个 body 各自排除隐藏 link 再取并集"变成一行 `obj_ids == self.robot_id`。
换来的是 `loadURDF` 上一个 flag。净删 95 行。

`get_foreground_gripper_points` 原来靠 `obj_ids == ghost_id` 认夹爪，现在按 link 过滤
（`gripper_links`，即名字不是 `panda_link*` 的那些）。

> 这次改动会改变渲染输出，所以 Stage 2/3 的产物需要重跑才和新代码同源。

In [ ]:
for f in glob.glob(os.path.join(REPO_DIR, "assets", "**", "_trimmed_*.urdf"), recursive=True):
  os.remove(f)
print("removed the trimmed copies this notebook's old-renderer cells wrote")